In [ ]:
import joblib
import optuna
import copy
from optuna.samplers import RandomSampler, TPESampler
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances, plot_slice, plot_parallel_coordinate
import numpy as np
import pandas as pd
from tabpfn import TabPFNClassifier, TabPFNRegressor
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.linear_model import SGDClassifier, SGDRegressor, LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor 
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score
from fastparquet import write
from fastparquet import ParquetFile
from pathlib import Path
import datetime as dt
import time
import pytz
import json
import os

In [ ]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

In [ ]:
oof_experiment_config = {
    "experiment": {
        "model": "xgboost",
        "type": "optuna",
        "num_trials": 50,
        "train_feature_groups": ["baseline/train.parq", "features/train.parq", "target_encode/train.parq"],
        "test_feature_groups": ["baseline/test.parq", "features/test.parq", "target_encode/test.parq"],
        "description": "xgb + clf + baseline + features + TE + optuna tune",
        "target": "classification",
    },

    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 0
    },

    "features": {
        "class_sample_weight": False
    },

    "params": {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 5000,
        "enable_categorical": True,
        "early_stopping_rounds": 100,
        "device": "cuda",
        "n_jobs": -1,
        "verbosity": 0
    },
    
    "fit_params": {
    }
}
oof_experiment_config["experiment"]["id"] = f"{dt_str}_{oof_experiment_config["experiment"]["model"]}"
experiment_config = copy.deepcopy(oof_experiment_config)

In [ ]:
def tune_params(trial):
    tree_method = trial.suggest_categorical("tree_method", ["hist", "approx"])
    booster = trial.suggest_categorical("booster", ["gbtree"])
    grow_policy = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])

    if grow_policy == "lossguide":
        max_depth = 0
        max_leaves = trial.suggest_int('max_leaves', 8, 512)
    else:
        max_depth = trial.suggest_int('max_depth', 2, 15)
        max_leaves = 0

    trial_params = {
        'max_depth': max_depth,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.4, log=True),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        "booster": booster,
        "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_float('min_child_weight', 1, 20, log=True),
        "max_bin": trial.suggest_int('max_bin', 32, 256, step=10),
        "tree_method": tree_method,
        "grow_policy": grow_policy
    }

    if grow_policy == "lossguide":
        trial_params["max_leaves"] = max_leaves

    return trial_params 

In [ ]:
sampler = RandomSampler()

In [ ]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../"

In [ ]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

In [ ]:
raw_train_df = pd.read_csv(f"{data_path}/raw/train.csv")
raw_train_id = raw_train_df["id"]
X_dfs, X_test_dfs = [], []

for fg in oof_experiment_config["experiment"]["train_feature_groups"]:
    file_path = f"{data_path}/{fg}"
    df = ParquetFile(file_path).to_pandas()
    X_dfs.append(df)

for fg in oof_experiment_config["experiment"]["test_feature_groups"]:
    file_path = f"{data_path}/{fg}"
    df = ParquetFile(file_path).to_pandas()
    X_test_dfs.append(df)

X = pd.concat(X_dfs, axis=1)
X_test = pd.concat(X_test_dfs, axis=1)
y = raw_train_df[target_column]

In [ ]:
def make_model(config, optuna_params):
    name = config["experiment"]["model"]
    target = config["experiment"]["target"]
    params = config["params"]

    model_dict = {
        "adaboost": (AdaBoostClassifier, AdaBoostRegressor),
        "gradientboost": (GradientBoostingClassifier, GradientBoostingRegressor),
        "catboost": (CatBoostClassifier, CatBoostRegressor), # TODO - fix missing col issue in catboost
        "xgboost": (XGBClassifier, XGBRegressor),
        "lightgbm": (LGBMClassifier, LGBMRegressor),
        "randomforest": (RandomForestClassifier, RandomForestRegressor),
        "extratrees": (ExtraTreesClassifier, ExtraTreesRegressor),
        "hgbc": (HistGradientBoostingClassifier, HistGradientBoostingRegressor),
        "knn": (KNeighborsClassifier, KNeighborsRegressor),
        "sgd": (SGDClassifier, SGDRegressor),
        "linear": (LogisticRegression, LinearRegression),
        "decisiontree": (DecisionTreeClassifier, DecisionTreeRegressor),
        "mlp": (MLPClassifier, MLPRegressor),
        "tabpfn": (TabPFNClassifier, TabPFNRegressor),
    }

    target_index = target == "regression"
    model_class = model_dict[name][target_index]

    return model_class(**params, **optuna_params)

In [ ]:
def oof_fit(config, model, X_train, y_train, X_valid, y_valid):
    name = config["experiment"]["model"]
    no_eval_models = ["adaboost", "gradientboost", "randomforest", "extratrees", "knn", "sgd", "linear", "decisiontree", "mlp", "tabpfn"]
    fit_params = config["fit_params"]
    
    if "features" in config and "class_sample_weight" in config["features"] and config["features"]["class_sample_weight"]:
        train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
    else:
        train_sample_weight = None

    if name in no_eval_models:
        model.fit(X_train, y_train, sample_weight=train_sample_weight)
    elif name == "hgbc":
        model.fit(X_train, y_train, X_val=X_valid, y_val=y_valid, sample_weight=train_sample_weight)
    elif name == "lightgbm":
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], sample_weight=train_sample_weight, **fit_params)
    else:
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], sample_weight=train_sample_weight, verbose=False)

In [ ]:
def predict(config, model, X_feat):
    task = config["experiment"]["target"]
    if task == "classification":
        return model.predict_proba(X_feat)[:, 1]
    else:
        return model.predict(X_feat)

In [ ]:
experiment_path = Path(output_path) / "experiments" / f"{dt_str}_optuna_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)

In [ ]:
def update_config(model, experiment_config):
    config = copy.deepcopy(experiment_config)
    params = config["params"]
    tree_param_keys = ["n_estimators", "max_iter"]

    best_iter = None
    if hasattr(model, "best_iteration_"):
        best_iter = model.best_iteration_
    elif hasattr(model, "best_iteration"):
        best_iter = model.best_iteration
    elif hasattr(model, "n_iter_"):
        n_iter_val = model.n_iter_
        
        if isinstance(n_iter_val, np.ndarray):
            if n_iter_val.size == 1:
                best_iter = n_iter_val.item()
            else:
                best_iter = int(n_iter_val.max())
        else:
            best_iter = int(n_iter_val)

    if best_iter is not None:
        for key in tree_param_keys:
            if key in params:
                params[key] = best_iter

    training_only_params = [
        "early_stopping_rounds",
        "early_stopping",
        "n_iter_no_change",
        "validation_fraction",
        "eval_set",
    ]

    for key in training_only_params:
        if key in params:
            del params[key]

    return config

In [ ]:
cv_config = experiment_config["cv"]
kf = StratifiedKFold(n_splits=cv_config["n_splits"], random_state=cv_config["random_state"], shuffle=cv_config["shuffle"])
y_cv = pd.Series(index=y.index, dtype=float, name=target_column)

train_index, valid_index = next(kf.split(X, y))

valid_ids = raw_train_id.iloc[valid_index].reset_index(drop=True)

X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

def objective(trial):
    local_dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")

    start_time = time.time()
    trial_params = tune_params(trial)
    model = make_model(oof_experiment_config, trial_params)

    oof_fit(
        oof_experiment_config,
        model,
        X_train,
        y_train,
        X_valid,
        y_valid
    )
    elapsed = time.time() - start_time
    
    y_pred = predict(oof_experiment_config, model, X_valid)
    y_pred_df = pd.concat([valid_ids, pd.Series(y_pred, name=target_column)], axis=1)
    valid_score = roc_auc_score(y_valid, y_pred)
    fold_scores = [round(valid_score, 5)]

    metrics = {
        "experiment": local_dt_str + f"_{experiment_config["experiment"]["model"]}",
        "model": f"{experiment_config["experiment"]["model"]}",
        "train_feature_groups": f"{experiment_config["experiment"]["train_feature_groups"]}",
        "cv": {
            "strategy": f"{experiment_config["cv"]["strategy"]}",
            "n_splits": experiment_config["cv"]["n_splits"],
            "random_state": experiment_config["cv"]["random_state"],
            "fold_scores": fold_scores,
            "mean": round(sum(fold_scores) / len(fold_scores), 5),
            "std": 0.0
        },
        "primary_metric": {
            "name": "auc",
            "value": round(valid_score, 5)
        },
        "training": {
            "duration_seconds": round(elapsed, 2)
        }
    }

    tune_experiment_config = copy.deepcopy(oof_experiment_config)
    tune_experiment_config["params"] = oof_experiment_config["params"] | trial_params

    full_config = update_config(model, tune_experiment_config)

    model_experiment_path = experiment_path / f"{experiment_config["experiment"]["model"]}_trial_{trial.number}"
    model_experiment_path.mkdir(parents=True, exist_ok=True)

    with open(model_experiment_path / "oof_config.json", "w") as f:
        json.dump(tune_experiment_config, f, indent=4)

    with open(model_experiment_path / "full_config.json", "w") as f:
        json.dump(full_config, f, indent=4)

    with open(model_experiment_path / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=4)

    y_pred_df.to_csv(model_experiment_path / "oof.csv", index=False)
    joblib.dump(model, model_experiment_path / f"{experiment_config["experiment"]["model"]}.pkl")

    return valid_score

In [ ]:
study = optuna.create_study(sampler=sampler, direction='maximize')
study.optimize(objective, n_trials=oof_experiment_config['experiment']['num_trials'])

In [ ]:
trial_df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
trial_df.to_csv(experiment_path / "optuna_trials.csv", index=False)
trial_df

In [ ]:
study.best_trial.params

In [ ]:
plot_optimization_history(study)

In [ ]:
plot_param_importances(study)

In [ ]:
plot_slice(study)